In [156]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

In [175]:
df = pd.read_csv("train.csv")
# df.head()

In [176]:
df.replace(['male', 'female'], [1, 0], inplace=True)

df["Title"] = df["Name"].str.extract(r',\s*([^.]*)\.')
df["Title"] = df["Title"].replace({
    "Mr": 0,
    "Miss": 1,
    "Mrs": 2,
    "Master": 3,
    "Don": 4,
    "Rev": 4,
    "Dr": 4,
    "Mme": 2,
    "Ms": 1,
    "Major": 4,
    "Lady": 4,
    "Sir": 4,
    "Mlle": 1,
    "Col": 4,
    "Capt": 4,
    "the Countess": 4,
    "Jonkheer": 4,
    "Dona": 4
})

df["Title"] = df["Title"].fillna(4)

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Fare"] = df["Fare"].fillna(df["Fare"].mean())
df["Embarked"] = df["Embarked"].fillna("S")

df["Embarked"] = df["Embarked"].map({
    "S": 0,
    "C": 1,
    "Q": 2
})
df["Sex_Pclass"] = df["Sex"] * df["Pclass"]

In [177]:
# df.info()

In [186]:
X_train = np.array(df[[
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "Title",
    "SibSp",
    "Parch",
    "FamilySize",
    "IsAlone",
    "Embarked",
    "Sex_Pclass"
]])
Y_train = np.array(df["Survived"])

X_train = X_train.astype(np.float32)
Y_train = Y_train.astype(np.int32)

mean = np.mean(X_train, axis=0)
std = np.std(X_train, axis=0)

X_train = (X_train - mean) / (std + 1e-8)

In [187]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(32,activation=tf.nn.relu),
    tf.keras.layers.Dense(32,activation=tf.nn.relu),
    tf.keras.layers.Dense(16,activation=tf.nn.relu),
    tf.keras.layers.Dense(2, activation=tf.nn.softmax)
])

In [188]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss=tf.keras.losses.sparse_categorical_crossentropy,
              metrics=['accuracy'])

In [189]:
model.fit(X_train,Y_train,epochs=100,validation_split=0.2)

Epoch 1/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6110 - loss: 0.7018 - val_accuracy: 0.6536 - val_loss: 0.5634
Epoch 2/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6573 - loss: 0.5836 - val_accuracy: 0.7654 - val_loss: 0.4953
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7444 - loss: 0.5476 - val_accuracy: 0.8212 - val_loss: 0.4679
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7963 - loss: 0.5235 - val_accuracy: 0.8212 - val_loss: 0.4462
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8104 - loss: 0.4989 - val_accuracy: 0.8324 - val_loss: 0.4283
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8174 - loss: 0.4739 - val_accuracy: 0.8436 - val_loss: 0.4051
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8230 - loss: 0.4554 - val_accuracy: 0.8547 - val_loss: 0.3922
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8202 - loss: 0.4431 - val_accuracy: 0.8659 - 

In [147]:
data_test = pd.read_csv("test.csv")

# جنسیت
data_test.replace(["male", "female"], [1, 0], inplace=True)

# عنوان
data_test["Title"] = data_test["Name"].str.extract(
    r',\s*([^.]*)\.'
)

data_test["Title"] = data_test["Title"].replace({
    "Mr": 0, "Miss": 1, "Mrs": 2, "Master": 3,
    "Don": 4, "Rev": 4, "Dr": 4, "Mme": 2,
    "Ms": 1, "Major": 4, "Lady": 4, "Sir": 4,
    "Mlle": 1, "Col": 4, "Capt": 4,
    "the Countess": 4, "Jonkheer": 4, "Dona": 4
}).fillna(4)

# اندازه خانواده و تنها بودن
data_test["FamilySize"] = (
    data_test["SibSp"] + data_test["Parch"] + 1
)
data_test["IsAlone"] = (
    data_test["FamilySize"] == 1
).astype(int)

# مقادیر خالی
data_test["Age"] = data_test["Age"].fillna(
    df["Age"].mean()
)
data_test["Fare"] = data_test["Fare"].fillna(
    df["Fare"].mean()
)
data_test["Embarked"] = data_test["Embarked"].fillna("S")

data_test["Embarked"] = data_test["Embarked"].map({
    "S": 0, "C": 1, "Q": 2
})

# ساخت X_test با همان ترتیب ویژگی‌های آموزش
X_test = np.array(data_test[[
    "Pclass", "Sex", "Age", "SibSp", "Parch",
    "Fare", "Title", "FamilySize", "IsAlone", "Embarked"
]], dtype=np.float32)

# نرمال‌سازی با میانگین و انحراف معیار آموزش
X_test = (X_test - mean) / (std + 1e-8)

gs_data = pd.read_csv("gender_submission.csv")
data_test["Survived"] = gs_data["Survived"]
Y_test = np.array(data_test["Survived"])

In [148]:
model.evaluate(X_test,Y_test)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8804 - loss: 0.4285  


[0.4284866452217102, 0.880382776260376]